# 第17周 · 课堂练习：DDPM 的零件、换算与 guidance

四组小练习，每组都有自查：**ᾱ 调度**、**前向加噪**、**ε / 数据 / 速度的换算**（与工具包对账）、**CFG 组合公式**。

约定（与讲义相同）：DDPM 中 $x = \sqrt{\bar\alpha}\,x_{\text{data}} + \sqrt{1-\bar\alpha}\,\varepsilon$，**时间 $\tau = 0$ 是数据**；FM 中 $x_t = t\,x_{\text{data}} + (1-t)\,\varepsilon$，**$t = 0$ 是噪声**；CFG 用 Ho & Salimans 原文约定 $(1+w)\cdot\text{cond} - w\cdot\text{uncond}$。代码里一律写 `x_data` 与 `eps`。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from zero2researcher import data, experiments, generative
from zero2researcher.checks import week17 as ck
from zero2researcher.checks import report

device = torch.device("cpu")      # 本练习全部是小张量运算
print("torch", torch.__version__, "| device:", device)

## E1 · ᾱ 调度

写 `alpha_bar(tau)`：DDPM 线性 β 调度（$10^{-4} \to 0.02$，$T = 1000$）的连续版 $\bar\alpha(\tau) = \exp(-(0.1\,\tau + 9.95\,\tau^2))$，对张量逐元素计算。自查会与原文的离散乘积 $\prod_{s \le t}(1 - \beta_s)$ 在 $\tau = t/T$ 处比较。

In [ ]:
def alpha_bar(tau):
    # 在这里写你的代码
    raise NotImplementedError


ck.alpha_bar(alpha_bar)

**E1b** · 两个数值问题，先估计，再用代码算：

1. $\tau = 0.5$（走了一半的 DDPM 时间）时，数据系数 $\sqrt{\bar\alpha}$ 是多少？还剩多少"数据"？
2. 与它**噪声水平相同**的 FM 时间 $t$ 是多少（用 `generative.ddpm_to_fm_time`）？和"直接翻转"的 $1 - \tau = 0.5$ 差多少？

把两个数分别存进 `data_coef` 与 `t_matched`。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print(f"τ = 0.5：数据系数 √ᾱ = {data_coef:.3f} | 噪声水平相同的 FM 时间 t = {t_matched:.3f}（直接翻转会得到 0.5）")
ck.half_time(data_coef, t_matched)

（一句话：为什么"DDPM 走了一半"的时候，对应的 FM 时间远小于 0.5？）

## E2 · 前向加噪 $q(x_t \mid x_0)$

写 `ddpm_forward(x_data, ab)`：在函数里抽 `eps = torch.randn_like(x_data)`，返回 `(x_t, eps)`，其中 $x_t = \sqrt{\bar\alpha}\,x_{\text{data}} + \sqrt{1-\bar\alpha}\,\varepsilon$。`ab` 是能广播的张量（例如 `(B, 1)`）。

In [ ]:
def ddpm_forward(x_data, ab):
    # 在这里写你的代码
    raise NotImplementedError


ck.ddpm_forward(ddpm_forward)

**E2b · "方差保持"（variance preserving）是什么意思？** 把月牙数据标准化成每个坐标方差为 1，然后在几个时间点上比较两条路径上 $x_t$ 的方差：

- DDPM：$\tau \in \{0, 0.25, 0.5, 1\}$，用你的 `ddpm_forward`；
- FM：$t \in \{1, 0.75, 0.5, 0\}$，$x_t = t\,x_{\text{data}} + (1-t)\,\varepsilon$（自己抽噪声）。

把每个时间点上 $x_t$ 两个坐标方差的平均分别存进列表 `var_ddpm` 与 `var_fm`。

In [ ]:
torch.manual_seed(0)
x = generative.sample_moons(50000)
x = (x - x.mean(0)) / x.std(0)                     # 每个坐标方差为 1
taus, ts = (0.0, 0.25, 0.5, 1.0), (1.0, 0.75, 0.5, 0.0)
# 在这里写你的代码
raise NotImplementedError
for tau, t, vd, vf in zip(taus, ts, var_ddpm, var_fm):
    print(f"DDPM τ = {tau:.2f}: Var = {vd:.3f}    |    FM t = {t:.2f}: Var = {vf:.3f}")

（一两句话：哪条路径的方差全程保持为 1？另一条在中间降到多少，为什么——用 $t^2 + (1-t)^2$ 解释？）

## E3 · ε、数据、速度的换算（与工具包对账）

写三个换算函数（讲义 S2 的换算表）：

- `eps_to_data_ddpm(eps, x_t, ab)`：DDPM 约定下，已知噪声预测求干净数据；
- `velocity_to_eps_fm(u, x_t, t)`：FM 约定下，已知速度 $u = x_{\text{data}} - \varepsilon$ 求噪声；
- `velocity_to_data_fm(u, x_t, t)`：FM 约定下，已知速度求干净数据。

提示：每个都是"两个方程、两个未知数"——一个是状态的定义 $x_t = a\,x_{\text{data}} + s\,\varepsilon$，另一个是预测量的定义。自查会把你的结果与 `generative.convert_prediction` 逐元素比较。

In [ ]:
def eps_to_data_ddpm(eps, x_t, ab):
    # 在这里写你的代码
    raise NotImplementedError


def velocity_to_eps_fm(u, x_t, t):
    # 在这里写你的代码
    raise NotImplementedError


def velocity_to_data_fm(u, x_t, t):
    # 在这里写你的代码
    raise NotImplementedError


ck.param_conversion(eps_to_data_ddpm, velocity_to_eps_fm, velocity_to_data_fm)

**E3b · 误差放大**。假设网络的 ε 预测有一个固定的小误差 $\delta = 0.01$（每个坐标）。用 FM 约定下"由 ε 求数据"的公式 $\hat x = (x_t - (1-t)\,\hat\varepsilon)/t$，数据估计的误差是 $\delta \cdot (1-t)/t$。对 $t \in \{0.9, 0.5, 0.1, 0.01\}$ 算出这个误差，存进列表 `amplified`。

In [ ]:
ts_q = (0.9, 0.5, 0.1, 0.01)
# 在这里写你的代码
raise NotImplementedError
for t, e in zip(ts_q, amplified):
    print(f"t = {t:<5}: ε 误差 0.01 → 数据估计误差 {e:.4f}（放大 {e / 0.01:.1f} 倍）")

（一两句话：这和"DDPM 模型换算成速度后、少步欧拉采样效果很差"有什么关系？FM 直接预测速度，为什么没有这个问题？）

## E4 · CFG 组合公式

写 `cfg_combine(pred_cond, pred_uncond, w)`：Ho & Salimans 原文约定，$w = 0$ 时原样返回条件预测。

In [ ]:
def cfg_combine(pred_cond, pred_uncond, w):
    # 在这里写你的代码
    raise NotImplementedError


ck.cfg_combine(cfg_combine)

**E4b · 两套 $w$ 的换算**。MIT 6.S184 的 Lab 用 $\tilde u = (1 - w_{\text{MIT}})\,u_\varnothing + w_{\text{MIT}}\,u_y$，试了 guidance $w_{\text{MIT}} \in \{1, 3, 5\}$。写 `mit_to_ho(w_mit)` 把它换成本课程（Ho 约定）的 $w$，并用数值验证：对随机的 `cond`、`uncond`，`cfg_combine(cond, uncond, mit_to_ho(w_mit))` 应等于 MIT 公式的结果。

In [ ]:
def mit_to_ho(w_mit):
    # 在这里写你的代码
    raise NotImplementedError


torch.manual_seed(0)
cond, uncond = torch.randn(5, 2), torch.randn(5, 2)
for w_mit in (1, 3, 5):
    mit = (1 - w_mit) * uncond + w_mit * cond
    ours = cfg_combine(cond, uncond, mit_to_ho(w_mit))
    print(f"w_MIT = {w_mit} → w_Ho = {mit_to_ho(w_mit)} | 两个公式的最大差 {(mit - ours).abs().max().item():.1e}")
report("w 约定换算", all(torch.allclose((1 - wm) * uncond + wm * cond, cfg_combine(cond, uncond, mit_to_ho(wm)))
                          for wm in (1, 3, 5)),
       "换算后两个公式结果不一致", "把 MIT 公式展开成 cond + (w_MIT − 1)·(cond − uncond)")

**E4c · 在 ε 上引导 = 在速度上引导**。Ho & Salimans 的公式是对 ε 写的，我们在 FM 里对速度用它。验证二者等价：给定同一个状态 `x_t`、时间 `t`，以及条件与无条件的 ε 预测，比较

- 路线 A：先在 ε 上组合，再用 `generative.convert_prediction` 换成速度；
- 路线 B：先把两个 ε 各自换成速度，再组合。

把两条路线的结果存进 `route_a`、`route_b`。

In [ ]:
torch.manual_seed(1)
x_t = torch.randn(100, 2)
t = torch.rand(100, 1) * 0.9 + 0.05
eps_cond, eps_uncond = torch.randn(100, 2), torch.randn(100, 2)
w = 2.0
# 在这里写你的代码
raise NotImplementedError
gap = (route_a - route_b).abs().max().item()
print(f"两条路线的最大差: {gap:.1e}")
report("ε 引导 = 速度引导", gap < 1e-4, f"两条路线应给出相同结果，最大差 {gap:.4f}",
       "两条路线都要用同一个 x_t 和 t；convert_prediction(pred, x_t, 'eps', 'velocity', t_fm=t)")

（一句话：为什么两条路线相等？提示：速度 = $(x_t - \hat\varepsilon)/t$，把 $(1+w) - w = 1$ 用上。）